# GarsonBot Colab Smoke-Test Fine-Tuning

This notebook is the first Colab-first training template for the `robot_waiter_ai` project.

Scope reminders:
- Turkish conversational waiter assistant only
- no ROS, navigation, SLAM, or motor-control work here
- deterministic baseline remains the reference system
- this is a smoke test, not a production recipe

Current confirmed dataset state:
- 154 seed Turkish examples
- 131 train records
- 23 validation records
- recommended first model: `Qwen/Qwen3-0.6B`


## 1. Runtime Check

Run this first. If no GPU is available, stop and switch the Colab runtime to GPU.


In [ ]:
import os
import platform
import subprocess
import sys

print('Python:', sys.version)
print('Platform:', platform.platform())

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError('No GPU runtime detected. Switch Colab to a GPU runtime before continuing.') from exc


## 2. Mount Drive And Clone Repo

Choose a persistent Drive folder for outputs. Update `REPO_URL` if your remote changes.


In [ ]:
USE_DRIVE = True
REPO_URL = 'https://github.com/your-org-or-user/Garson-bot.git'
REPO_DIR = '/content/Garson-bot'
RUN_NAME = 'run_001_qwen3_0p6b_smoke'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/garsonbot_runs'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already present:', REPO_DIR)

%cd /content/Garson-bot


## 3. Repo And Dataset Sanity Check

This verifies we are pointing at the right repo copy before installing the heavy training stack.


In [ ]:
from pathlib import Path
import json

train_path = Path('robot_waiter_ai/datasets/processed/waiter_sft_train.jsonl')
valid_path = Path('robot_waiter_ai/datasets/processed/waiter_sft_valid.jsonl')
eval_path = Path('robot_waiter_ai/evals/evaluation_cases.yaml')

for path in [train_path, valid_path, eval_path]:
    if not path.exists():
        raise FileNotFoundError(f'Missing expected file: {path}')
    print('Found:', path)

def load_jsonl(path: Path):
    rows = []
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(train_path)
valid_rows = load_jsonl(valid_path)

print('Train rows:', len(train_rows))
print('Valid rows:', len(valid_rows))

sample = train_rows[0]
assert 'messages' in sample and isinstance(sample['messages'], list)
assert sample['messages'][-1]['role'] == 'assistant'
print('Sample metadata:', sample.get('metadata', {}))
print('Sample messages:', sample['messages'])


## 4. Install Training Dependencies In Colab Only

These packages are intentionally not part of the local repository environment.


In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes sentencepiece


## 5. Training Configuration

These values match the repo's current planning defaults for the first smoke test.


In [ ]:
from dataclasses import asdict, dataclass

@dataclass
class RunConfig:
    base_model_name_or_path: str = 'Qwen/Qwen3-0.6B'
    output_dir: str = f'{DRIVE_OUTPUT_ROOT}/{RUN_NAME}' if USE_DRIVE else f'/content/{RUN_NAME}'
    train_file: str = str(train_path)
    valid_file: str = str(valid_path)
    method: str = 'qlora'
    epochs: int = 3
    learning_rate: float = 2e-4
    batch_size: int = 2
    gradient_accumulation_steps: int = 8
    max_seq_length: int = 1024
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    eval_steps: int = 10
    save_steps: int = 25
    seed: int = 42

config = RunConfig()
asdict(config)


## 6. Load Model, Tokenizer, And Dataset

This cell uses 4-bit loading for a conservative QLoRA smoke test.


In [ ]:
import json
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

def load_rows(path: str):
    rows = []
    with open(path, 'r', encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows

train_rows = load_rows(config.train_file)
valid_rows = load_rows(config.valid_file)

def render_chat(row):
    text = tokenizer.apply_chat_template(
        row['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {
        'text': text,
        'messages': row['messages'],
        'metadata': row.get('metadata', {}),
    }

train_dataset = Dataset.from_list([render_chat(row) for row in train_rows])
valid_dataset = Dataset.from_list([render_chat(row) for row in valid_rows])

print(train_dataset)
print(valid_dataset)
print(train_dataset[0]['text'][:1000])


## 7. Configure LoRA And Trainer

If the base model uses different preferred target modules, adjust `target_modules` after inspecting a known-good recipe for that exact model revision.


In [ ]:
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj', 'gate_proj'],
)

training_args = SFTConfig(
    output_dir=config.output_dir,
    num_train_epochs=config.epochs,
    learning_rate=config.learning_rate,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    max_seq_length=config.max_seq_length,
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_strategy='steps',
    seed=config.seed,
    report_to='none',
    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
    fp16=not (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8),
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)


## 8. Train

This is the first real training step. Expect this to take time and watch for out-of-memory issues.


In [ ]:
train_result = trainer.train()
train_result


## 9. Save Adapter, Tokenizer, And Metrics

Save adapter-only artifacts first. That keeps the first smoke-test lightweight.


In [ ]:
import json
from pathlib import Path

output_dir = Path(config.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(output_dir / 'adapter'))
tokenizer.save_pretrained(str(output_dir / 'tokenizer'))

metrics = train_result.metrics
with (output_dir / 'train_metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(metrics, handle, ensure_ascii=False, indent=2)

print('Saved artifacts under:', output_dir)


## 10. Manual Turkish Smoke Tests

These prompts help us inspect role consistency, menu faithfulness, and allergy caution before deeper evaluation.


In [ ]:
smoke_prompts = [
    'Merhaba',
    'Bir mercimek corbasi ve bir ayran alabilir miyim?',
    'Fistik alerjim var, ne onerirsiniz?',
    'Menude olmayan sushi var mi?',
]

def generate_response(user_text: str, max_new_tokens: int = 128):
    messages = [
        {
            'role': 'system',
            'content': 'Sen nazik, dikkatli ve menüye sadik bir restoranda garson robot asistanisin. Her zaman Türkçe cevap ver.',
        },
        {'role': 'user', 'content': user_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        generated = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tokenizer.decode(generated[0], skip_special_tokens=True)
    return decoded

smoke_rows = []
for prompt in smoke_prompts:
    response = generate_response(prompt)
    smoke_rows.append({'prompt': prompt, 'response': response})
    print('\nPROMPT:', prompt)
    print('RESPONSE:', response)


## 11. Save Smoke Outputs

This creates a simple artifact we can inspect later alongside the adapter checkpoint.


In [ ]:
smoke_output_path = output_dir / 'smoke_generations.jsonl'
with smoke_output_path.open('w', encoding='utf-8') as handle:
    for row in smoke_rows:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')

print('Saved smoke outputs to:', smoke_output_path)


## 12. Next Evaluation Hand-Off

After the first successful run, the next step is to generate benchmark outputs in this JSONL shape:

```json
{"case_id": "eval_001", "response": "..."}
```

That file can then be scored with the existing repo adapter:

```powershell
.venv\Scripts\python.exe -m robot_waiter_ai.evals.generated_output_adapter --outputs path\to\generated_outputs.jsonl
```

For notebook V1, manual smoke prompts are enough. Benchmark generation can be the next iteration.
